In [ ]:
#!/usr/bin/env python3
"""
FruitNet-GCS fixed: one-file COCO pretraining + public benchmark pipeline.

Validated target environment:
    ultralytics==8.4.60

Main properties:
- Creates a stable helper package at models/fruitnet_gcs_fixed/.
- Uses a real SIoU DetectionTrainer, not a global monkey patch.
- Uses only names/folders ending in or containing "fixed".
- Never overwrites or adopts legacy FruitNet-GCS traces.
- Audits datasets and the actual criterion used at runtime.
- Saves last.pt every epoch, resumes from a resumable last.pt after crash.
- Skips only when marker, job signature, checkpoint SHA-256, results.csv,
  and loss_audit.json all agree.
- Quarantines stale, non-resumable run folders instead of deleting them.
- Pretrains on COCO, then fine-tunes/evaluates on MinneApple and GWHD.

Typical run:
    export FRUITNET_PROJECT_ROOT=/home/diy-hus/fruitnet-chili-yield
    export FRUITNET_DATA_ROOT=/mnt/f/fruitnet-chili-yield-data
    export FRUITNET_OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
    export DEVICE=0
    python fruitnet_gcs_fixed_pretrain_benchmark.py

Useful controls:
    COCO_EPOCHS=50
    BENCHMARK_EPOCHS=100
    IMG_SIZE=640
    COCO_BATCH=4
    BENCHMARK_BATCH=8
    WORKERS=8
    SEED=42
    COCO_PATIENCE=20
    BENCHMARK_PATIENCE=30
    SAVE_PERIOD=1
    PLOTS=1
    BENCHMARK_DATASETS=minneapple,gwhd
    BENCHMARK_EVAL_SPLIT=auto   # auto: independent test if available, else val
    RUN_PRETRAIN=1
    RUN_BENCHMARK=1
    RUN_EVAL=1
    FORCE_PRETRAIN=0
    FORCE_BENCHMARK=0
    FORCE_EVAL=0
    STRICT_ULTRALYTICS_VERSION=1
    STRICT_DATASET_AUDIT=1
"""

from __future__ import annotations

import hashlib
import importlib
import inspect
import json
import logging
import os
import platform
import shutil
import sys
import tempfile
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

try:
    import pandas as pd
    import yaml
except ImportError as exc:
    raise RuntimeError("Install dependencies first: pip install pandas PyYAML") from exc


# =============================================================================
# 0. CONSTANTS AND ENVIRONMENT
# =============================================================================

EXPECTED_ULTRALYTICS_VERSION = "8.4.60"
PIPELINE_SCHEMA_VERSION = 3
MODEL_KEY = "fruitnet_gcs_fixed"
EXPECTED_LOSS = "siou"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def env_bool(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def find_project_root() -> Path:
    override = os.environ.get("FRUITNET_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"FRUITNET_PROJECT_ROOT does not exist: {root}")
        return root

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if candidate.name == "fruitnet-chili-yield":
            return candidate
        if (candidate / "models").exists() and (candidate / "configs").exists():
            return candidate
    raise RuntimeError(
        "Cannot locate project root. Set FRUITNET_PROJECT_ROOT=/path/to/fruitnet-chili-yield"
    )


def default_data_root(project_root: Path) -> Path:
    raw = (
        os.environ.get("FRUITNET_DATA_ROOT", "").strip()
        or os.environ.get("DATA_ROOT", "").strip()
    )
    if raw:
        return Path(raw).expanduser().resolve()
    wsl = Path("/mnt/f/fruitnet-chili-yield-data")
    return wsl if wsl.parent.exists() else (project_root / "data").resolve()


def default_output_root(project_root: Path) -> Path:
    raw = (
        os.environ.get("FRUITNET_OUTPUT_ROOT", "").strip()
        or os.environ.get("OUTPUT_ROOT", "").strip()
    )
    if raw:
        return Path(raw).expanduser().resolve()
    wsl = Path("/mnt/f/fruitnet-chili-yield-outputs")
    return wsl if wsl.parent.exists() else (project_root / "outputs").resolve()


PROJECT_ROOT = find_project_root()
DATA_ROOT = default_data_root(PROJECT_ROOT)
OUTPUT_ROOT = default_output_root(PROJECT_ROOT)
FIXED_OUTPUT_ROOT = OUTPUT_ROOT / "fruitnet_gcs_fixed"

MODEL_PACKAGE_DIR = PROJECT_ROOT / "models" / "fruitnet_gcs_fixed"
MODEL_YAML = PROJECT_ROOT / "configs" / "models" / "fruitnet_gcs_fixed.yaml"
FIXED_DATA_CONFIG_DIR = PROJECT_ROOT / "configs" / "data" / "fixed"
MODEL_AUDIT_DIR = PROJECT_ROOT / "results" / "model_registry_fixed"

COCO_RUNS_ROOT = FIXED_OUTPUT_ROOT / "runs" / "coco_pretrain_fixed"
COCO_ARTIFACT_ROOT = FIXED_OUTPUT_ROOT / "artifacts" / "pretrained_fixed"
COCO_RESULT_ROOT = FIXED_OUTPUT_ROOT / "results" / "coco_pretrain_fixed"
COCO_MARKER_ROOT = COCO_RESULT_ROOT / "job_markers_fixed"
COCO_ARTIFACT = COCO_ARTIFACT_ROOT / "fruitnet_gcs_fixed_coco_best.pt"

BENCH_RUNS_ROOT = FIXED_OUTPUT_ROOT / "runs" / "public_benchmark_fixed"
BENCH_EVAL_RUNS_ROOT = FIXED_OUTPUT_ROOT / "runs" / "public_benchmark_eval_fixed"
BENCH_ARTIFACT_ROOT = FIXED_OUTPUT_ROOT / "artifacts" / "public_benchmark_fixed"
BENCH_RESULT_ROOT = FIXED_OUTPUT_ROOT / "results" / "public_benchmark_fixed"
BENCH_MARKER_ROOT = BENCH_RESULT_ROOT / "job_markers_fixed"
BENCH_METRICS_ROOT = BENCH_RESULT_ROOT / "metrics_fixed"

LOG_DIR = FIXED_OUTPUT_ROOT / "logs_fixed"
LOG_PATH = LOG_DIR / f"fruitnet_gcs_fixed_{datetime.now():%Y%m%d_%H%M%S}.log"


# =============================================================================
# 1. SAFE I/O, HASHING, LOGGING
# =============================================================================


def ensure_dirs(paths: Iterable[Path]) -> None:
    for path in paths:
        path.mkdir(parents=True, exist_ok=True)


def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding=encoding) as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)


def atomic_write_json(path: Path, payload: Any) -> None:
    atomic_write_text(path, json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n")


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    try:
        frame.to_csv(temp_name, index=False)
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)


def append_jsonl(path: Path, payload: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(dict(payload), ensure_ascii=False, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def atomic_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{dst.name}.", suffix=".tmp", dir=dst.parent)
    os.close(fd)
    try:
        shutil.copy2(src, temp_name)
        os.replace(temp_name, dst)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(payload: Any) -> str:
    text = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path: Path, default: Any = None) -> Any:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def read_yaml(path: Path, default: Any = None) -> Any:
    try:
        return yaml.safe_load(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def setup_logging() -> None:
    ensure_dirs([LOG_DIR])
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
        force=True,
    )


def write_failure(path: Path, job_id: str, exc: BaseException, extra: Mapping[str, Any] | None = None) -> None:
    payload: dict[str, Any] = {
        "time": now_iso(),
        "status": "failed",
        "job_id": job_id,
        "error": repr(exc),
        "traceback": traceback.format_exc(),
    }
    if extra:
        payload.update(extra)
    atomic_write_json(path, payload)


def archive_stale_run(run_dir: Path, reason: str) -> Path | None:
    if not run_dir.exists() or not any(run_dir.iterdir()):
        return None
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    target = run_dir.parent / f"{run_dir.name}.stale_fixed_{stamp}"
    index = 1
    while target.exists():
        target = run_dir.parent / f"{run_dir.name}.stale_fixed_{stamp}_{index}"
        index += 1
    os.replace(run_dir, target)
    atomic_write_json(
        target / "stale_reason_fixed.json",
        {"archived_at": now_iso(), "original_run_dir": str(run_dir), "reason": reason},
    )
    return target


# =============================================================================
# 2. WRITE STABLE CUSTOM MODULES AND MODEL YAML
# =============================================================================

COORD_ATTENTION_SOURCE = r'''from __future__ import annotations

import torch
import torch.nn as nn


class CoordAttFixed(nn.Module):
    """Coordinate Attention with unchanged channel count."""

    def __init__(self, channels: int, reduction: int = 32):
        super().__init__()
        self.channels = int(channels)
        mip = max(8, self.channels // int(reduction))
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(self.channels, mip, 1, 1, 0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU(inplace=True)
        self.conv_h = nn.Conv2d(mip, self.channels, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, self.channels, 1, 1, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        _, _, height, width = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        merged = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(merged, [height, width], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return identity * self.conv_h(x_h).sigmoid() * self.conv_w(x_w).sigmoid()
'''

SIOU_LOSS_SOURCE = r'''from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss
from ultralytics.utils.tal import bbox2dist


def bbox_iou_xyxy_fixed(box1: torch.Tensor, box2: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    lt = torch.maximum(box1[..., :2], box2[..., :2])
    rb = torch.minimum(box1[..., 2:], box2[..., 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    area1 = (box1[..., 2] - box1[..., 0]).clamp(min=0) * (box1[..., 3] - box1[..., 1]).clamp(min=0)
    area2 = (box2[..., 2] - box2[..., 0]).clamp(min=0) * (box2[..., 3] - box2[..., 1]).clamp(min=0)
    return inter / (area1 + area2 - inter + eps)


def siou_loss_xyxy_fixed(
    pred: torch.Tensor,
    target: torch.Tensor,
    eps: float = 1e-7,
    theta: float = 4.0,
) -> torch.Tensor:
    """Unreduced SIoU regression loss for aligned xyxy boxes."""
    iou = bbox_iou_xyxy_fixed(pred, target, eps=eps)
    pred_center = (pred[..., :2] + pred[..., 2:]) * 0.5
    target_center = (target[..., :2] + target[..., 2:]) * 0.5
    delta = target_center - pred_center

    pred_w = (pred[..., 2] - pred[..., 0]).clamp(min=eps)
    pred_h = (pred[..., 3] - pred[..., 1]).clamp(min=eps)
    target_w = (target[..., 2] - target[..., 0]).clamp(min=eps)
    target_h = (target[..., 3] - target[..., 1]).clamp(min=eps)

    enclosing_w = (
        torch.maximum(pred[..., 2], target[..., 2])
        - torch.minimum(pred[..., 0], target[..., 0])
    ).clamp(min=eps)
    enclosing_h = (
        torch.maximum(pred[..., 3], target[..., 3])
        - torch.minimum(pred[..., 1], target[..., 1])
    ).clamp(min=eps)

    sigma = torch.sqrt(delta[..., 0].square() + delta[..., 1].square() + eps)
    sin_alpha_1 = delta[..., 0].abs() / sigma
    sin_alpha_2 = delta[..., 1].abs() / sigma
    threshold = math.sqrt(2.0) / 2.0
    sin_alpha = torch.where(sin_alpha_1 > threshold, sin_alpha_2, sin_alpha_1).clamp(0.0, 1.0 - 1e-6)
    angle_cost = torch.cos(torch.asin(sin_alpha) * 2.0 - math.pi / 2.0)

    gamma = 2.0 - angle_cost
    rho_x = (delta[..., 0] / enclosing_w).square()
    rho_y = (delta[..., 1] / enclosing_h).square()
    distance_cost = 2.0 - torch.exp(-gamma * rho_x) - torch.exp(-gamma * rho_y)

    omega_w = (pred_w - target_w).abs() / torch.maximum(pred_w, target_w)
    omega_h = (pred_h - target_h).abs() / torch.maximum(pred_h, target_h)
    shape_cost = (1.0 - torch.exp(-omega_w)).pow(theta) + (1.0 - torch.exp(-omega_h)).pow(theta)

    result = 1.0 - iou + 0.5 * (distance_cost + shape_cost)
    return torch.nan_to_num(result, nan=1.0, posinf=2.0, neginf=0.0)


class SIoUBboxLossFixed(BboxLoss):
    """Ultralytics BboxLoss with SIoU replacing CIoU while preserving DFL."""

    def forward(
        self,
        pred_dist: torch.Tensor,
        pred_bboxes: torch.Tensor,
        anchor_points: torch.Tensor,
        target_bboxes: torch.Tensor,
        target_scores: torch.Tensor,
        target_scores_sum: torch.Tensor,
        fg_mask: torch.Tensor,
        imgsz: torch.Tensor,
        stride: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        per_box = siou_loss_xyxy_fixed(pred_bboxes[fg_mask], target_bboxes[fg_mask]).unsqueeze(-1)
        loss_iou = (per_box * weight).sum() / target_scores_sum

        if self.dfl_loss:
            target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
            loss_dfl = self.dfl_loss(
                pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max),
                target_ltrb[fg_mask],
            ) * weight
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            target_ltrb = bbox2dist(anchor_points, target_bboxes)
            target_ltrb = target_ltrb * stride
            target_ltrb[..., 0::2] /= imgsz[1]
            target_ltrb[..., 1::2] /= imgsz[0]
            pred_dist = pred_dist * stride
            pred_dist[..., 0::2] /= imgsz[1]
            pred_dist[..., 1::2] /= imgsz[0]
            loss_dfl = (
                F.l1_loss(pred_dist[fg_mask], target_ltrb[fg_mask], reduction="none")
                .mean(-1, keepdim=True)
                .mul(weight)
                .sum()
                / target_scores_sum
            )
        return loss_iou, loss_dfl


class SIoUDetectionLossFixed(v8DetectionLoss):
    loss_mode = "siou"

    def __init__(self, model: torch.nn.Module, tal_topk: int = 10, tal_topk2: int | None = None):
        super().__init__(model, tal_topk=tal_topk, tal_topk2=tal_topk2)
        self.bbox_loss = SIoUBboxLossFixed(self.reg_max).to(self.device)
'''

SIOU_TRAINER_SOURCE = r'''from __future__ import annotations

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from .siou_loss import SIoUDetectionLossFixed


class SIoUDetectionModelFixed(DetectionModel):
    loss_mode = "siou"

    def init_criterion(self):
        return SIoUDetectionLossFixed(self)


class SIoUDetectionTrainerFixed(DetectionTrainer):
    loss_mode = "siou"

    def get_model(self, cfg: str | None = None, weights: str | None = None, verbose: bool = True):
        model = SIoUDetectionModelFixed(
            cfg,
            nc=self.data["nc"],
            ch=self.data["channels"],
            verbose=verbose and RANK == -1,
        )
        if weights:
            model.load(weights)
        return model
'''

REGISTER_SOURCE = r'''from __future__ import annotations


def register_fruitnet_gcs_fixed_modules():
    from .coord_attention import CoordAttFixed
    import ultralytics.nn.tasks as tasks

    tasks.CoordAttFixed = CoordAttFixed
    try:
        import ultralytics.nn.modules as modules
        modules.CoordAttFixed = CoordAttFixed
    except Exception:
        pass
    return {"CoordAttFixed": CoordAttFixed}
'''

INIT_SOURCE = r'''from .coord_attention import CoordAttFixed
from .siou_loss import SIoUBboxLossFixed, SIoUDetectionLossFixed, siou_loss_xyxy_fixed
from .siou_trainer import SIoUDetectionModelFixed, SIoUDetectionTrainerFixed
from .register import register_fruitnet_gcs_fixed_modules
'''

MODEL_YAML_SOURCE = r'''# FruitNet-GCS fixed
# Architecture: GhostConv + Coordinate Attention.
# Training loss: SIoU through SIoUDetectionTrainerFixed.
# Parameters/GFLOPs are expected to match the GC architecture; loss does not alter inference complexity.
nc: 80
backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, GhostConv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, CoordAttFixed, [64]]
  - [-1, 1, GhostConv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, CoordAttFixed, [128]]
  - [-1, 1, GhostConv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, CoordAttFixed, [256]]
  - [-1, 1, GhostConv, [512, 3, 2]]
  - [-1, 2, C2f, [512, True]]
  - [-1, 1, SPPF, [512, 5]]
  - [-1, 1, CoordAttFixed, [512]]
head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]
  - [-1, 1, CoordAttFixed, [256]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C2f, [128]]
  - [-1, 1, CoordAttFixed, [128]]
  - [-1, 1, GhostConv, [128, 3, 2]]
  - [[-1, 17], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]
  - [-1, 1, CoordAttFixed, [256]]
  - [-1, 1, GhostConv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C2f, [512]]
  - [-1, 1, CoordAttFixed, [512]]
  - [[21, 25, 29], 1, Detect, [nc]]
'''


def install_fixed_model_files() -> None:
    ensure_dirs([MODEL_PACKAGE_DIR, MODEL_YAML.parent, MODEL_AUDIT_DIR])
    models_init = PROJECT_ROOT / "models" / "__init__.py"
    if not models_init.exists():
        atomic_write_text(models_init, "")
    atomic_write_text(MODEL_PACKAGE_DIR / "__init__.py", INIT_SOURCE)
    atomic_write_text(MODEL_PACKAGE_DIR / "coord_attention.py", COORD_ATTENTION_SOURCE)
    atomic_write_text(MODEL_PACKAGE_DIR / "siou_loss.py", SIOU_LOSS_SOURCE)
    atomic_write_text(MODEL_PACKAGE_DIR / "siou_trainer.py", SIOU_TRAINER_SOURCE)
    atomic_write_text(MODEL_PACKAGE_DIR / "register.py", REGISTER_SOURCE)
    atomic_write_text(MODEL_YAML, MODEL_YAML_SOURCE)
    importlib.invalidate_caches()


# =============================================================================
# 3. DATASET CONFIG AND AUDIT
# =============================================================================

COCO_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog",
    "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite",
    "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
    "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich",
    "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote",
    "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book",
    "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush",
]


def resolve_yaml_base(yaml_path: Path, payload: Mapping[str, Any]) -> Path:
    raw = Path(str(payload.get("path", yaml_path.parent))).expanduser()
    return raw.resolve() if raw.is_absolute() else (yaml_path.parent / raw).resolve()


def resolve_split_from_payload(yaml_path: Path, payload: Mapping[str, Any], split: str) -> Path | None:
    value = payload.get(split)
    if value is None or isinstance(value, list):
        return None
    split_path = Path(str(value)).expanduser()
    if split_path.is_absolute():
        return split_path.resolve()
    return (resolve_yaml_base(yaml_path, payload) / split_path).resolve()


def nonempty_image_dir(path: Path | None) -> bool:
    return bool(path and path.exists() and any(p.is_file() and p.suffix.lower() in IMAGE_EXTS for p in path.rglob("*")))


def infer_dataset_root(dataset: str) -> Path:
    env_name = {
        "coco": "COCO_YOLO_ROOT",
        "minneapple": "MINNEAPPLE_YOLO_ROOT",
        "gwhd": "GWHD_YOLO_ROOT",
    }[dataset]
    raw = os.environ.get(env_name, "").strip()
    if raw:
        return Path(raw).expanduser().resolve()
    return {
        "coco": DATA_ROOT / "processed" / "coco_yolo",
        "minneapple": DATA_ROOT / "processed" / "minneapple_yolo",
        "gwhd": DATA_ROOT / "processed" / "gwhd_yolo",
    }[dataset].resolve()


def build_fixed_dataset_yaml(dataset: str) -> tuple[Path, str]:
    """Create a new fixed YAML without modifying the old YAML.

    Returns (yaml_path, evaluation_split). For benchmark datasets, auto uses an
    independent test split only when it exists and differs from val; otherwise val.
    """
    dataset = dataset.lower()
    source_yaml = PROJECT_ROOT / "configs" / "data" / f"{dataset}.yaml"
    source_payload = read_yaml(source_yaml, {}) or {}
    root = resolve_yaml_base(source_yaml, source_payload) if source_payload else infer_dataset_root(dataset)

    if dataset == "coco":
        payload: dict[str, Any] = {
            "path": str(root),
            "train": source_payload.get("train", "images/train2017"),
            "val": source_payload.get("val", "images/val2017"),
            "names": source_payload.get("names", {i: name for i, name in enumerate(COCO_NAMES)}),
        }
        eval_split = "val"
    else:
        default_name = "apple" if dataset == "minneapple" else "wheat_head"
        payload = {
            "path": str(root),
            "train": source_payload.get("train", "images/train"),
            "val": source_payload.get("val", "images/val"),
            "names": source_payload.get("names", {0: default_name}),
        }

        old_test = resolve_split_from_payload(source_yaml, source_payload, "test") if source_payload else None
        old_val = resolve_split_from_payload(source_yaml, source_payload, "val") if source_payload else None
        inferred_test = root / "images" / "test"
        independent_test: Path | None = None
        if nonempty_image_dir(old_test) and (old_val is None or old_test.resolve() != old_val.resolve()):
            independent_test = old_test
        elif nonempty_image_dir(inferred_test):
            independent_test = inferred_test

        requested = os.environ.get("BENCHMARK_EVAL_SPLIT", "auto").strip().lower()
        if requested not in {"auto", "val", "test"}:
            raise ValueError("BENCHMARK_EVAL_SPLIT must be auto, val, or test")
        if requested == "test" and independent_test is None:
            raise RuntimeError(
                f"{dataset}: BENCHMARK_EVAL_SPLIT=test but no independent test directory exists."
            )
        if requested == "val":
            eval_split = "val"
        elif independent_test is not None:
            payload["test"] = str(independent_test)
            eval_split = "test"
        else:
            eval_split = "val"
            logging.warning(
                "%s has no independent test split. Final benchmark is reported on val and is not an independent test result.",
                dataset,
            )

    fixed_yaml = FIXED_DATA_CONFIG_DIR / f"{dataset}_fixed.yaml"
    atomic_write_text(fixed_yaml, yaml.safe_dump(payload, sort_keys=False, allow_unicode=True))
    return fixed_yaml, eval_split


def image_files(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def label_root_for(image_root: Path) -> Path:
    parts = list(image_root.parts)
    if "images" in parts:
        index = len(parts) - 1 - parts[::-1].index("images")
        return Path(*parts[:index], "labels", *parts[index + 1 :])
    return image_root.parent.parent / "labels" / image_root.name


def tree_fingerprint(root: Path, suffixes: set[str]) -> dict[str, Any]:
    rows: list[tuple[str, int, int]] = []
    if root.exists():
        for path in sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in suffixes):
            stat = path.stat()
            rows.append((path.relative_to(root).as_posix(), stat.st_size, stat.st_mtime_ns))
    return {
        "root": str(root),
        "file_count": len(rows),
        "total_bytes": sum(row[1] for row in rows),
        "fingerprint": stable_hash(rows),
    }


def audit_yolo_dataset(data_yaml: Path, required_splits: Sequence[str]) -> dict[str, Any]:
    payload = read_yaml(data_yaml, {}) or {}
    names = payload.get("names", {})
    valid_class_ids = set(range(len(names))) if isinstance(names, list) else {int(key) for key in names.keys()}
    strict = env_bool("STRICT_DATASET_AUDIT", True)

    splits: dict[str, Any] = {}
    keys_by_split: dict[str, set[str]] = {}
    errors: list[str] = []

    for split in required_splits:
        image_root = resolve_split_from_payload(data_yaml, payload, split)
        if image_root is None or not image_root.exists():
            errors.append(f"Missing split {split}: {image_root}")
            continue
        labels_root = label_root_for(image_root)
        images = image_files(image_root)
        relative_keys: set[str] = set()
        missing_labels = 0
        invalid_lines = 0
        boxes = 0
        classes_seen: set[int] = set()

        for image in images:
            relative = image.relative_to(image_root)
            relative_keys.add(relative.with_suffix("").as_posix().lower())
            label = (labels_root / relative).with_suffix(".txt")
            if not label.exists():
                missing_labels += 1
                continue
            for line in label.read_text(encoding="utf-8", errors="replace").splitlines():
                if not line.strip():
                    continue
                fields = line.split()
                try:
                    if len(fields) != 5:
                        raise ValueError("expected five YOLO fields")
                    cls = int(float(fields[0]))
                    coords = [float(value) for value in fields[1:]]
                    if cls not in valid_class_ids:
                        raise ValueError(f"class id {cls} not declared in names")
                    if not all(0.0 <= value <= 1.0 for value in coords):
                        raise ValueError("coordinates outside [0,1]")
                    if coords[2] <= 0 or coords[3] <= 0:
                        raise ValueError("non-positive width/height")
                    boxes += 1
                    classes_seen.add(cls)
                except Exception:
                    invalid_lines += 1

        splits[split] = {
            "image_root": str(image_root),
            "label_root": str(labels_root),
            "images": len(images),
            "boxes": boxes,
            "missing_label_files": missing_labels,
            "invalid_label_lines": invalid_lines,
            "classes_seen": sorted(classes_seen),
            "image_tree": tree_fingerprint(image_root, IMAGE_EXTS),
            "label_tree": tree_fingerprint(labels_root, {".txt"}),
        }
        keys_by_split[split] = relative_keys
        if not images:
            errors.append(f"Split {split} contains no images")
        if invalid_lines:
            errors.append(f"Split {split} contains {invalid_lines} invalid label lines")
        if strict and missing_labels:
            errors.append(f"Split {split} has {missing_labels} images without label files")

    overlaps: dict[str, int] = {}
    split_names = list(keys_by_split)
    for index, first in enumerate(split_names):
        for second in split_names[index + 1 :]:
            overlap = len(keys_by_split[first] & keys_by_split[second])
            overlaps[f"{first}__{second}"] = overlap
            if overlap:
                errors.append(f"Splits {first} and {second} overlap on {overlap} relative image keys")

    audit = {
        "schema_version": PIPELINE_SCHEMA_VERSION,
        "created_at": now_iso(),
        "data_yaml": str(data_yaml.resolve()),
        "data_yaml_sha256": sha256_file(data_yaml),
        "names": names,
        "splits": splits,
        "overlaps": overlaps,
        "strict_missing_labels": strict,
        "valid": not errors,
        "errors": errors,
    }
    audit["manifest_hash"] = stable_hash(
        {key: value for key, value in audit.items() if key not in {"created_at", "manifest_hash"}}
    )
    return audit


# =============================================================================
# 4. CHECKPOINT, RESULT, MARKER, CALLBACK AUDIT
# =============================================================================


def register_fixed_modules() -> None:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    from models.fruitnet_gcs_fixed.register import register_fruitnet_gcs_fixed_modules

    register_fruitnet_gcs_fixed_modules()


def validate_checkpoint(path: Path, require_resumable: bool = False) -> dict[str, Any]:
    result: dict[str, Any] = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "loadable": False,
        "resumable": False,
        "sha256": "",
        "error": "",
    }
    if not path.exists() or path.stat().st_size < 1024:
        result["error"] = "missing or implausibly small checkpoint"
        return result
    try:
        register_fixed_modules()
        import torch

        try:
            checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            checkpoint = torch.load(path, map_location="cpu")
        result["loadable"] = bool(
            isinstance(checkpoint, dict)
            and (checkpoint.get("model") is not None or checkpoint.get("ema") is not None)
        )
        if isinstance(checkpoint, dict):
            result["epoch"] = checkpoint.get("epoch")
            result["checkpoint_version"] = checkpoint.get("version")
            result["resumable"] = bool(
                checkpoint.get("epoch", -1) is not None
                and int(checkpoint.get("epoch", -1)) >= 0
                and checkpoint.get("optimizer") is not None
            )
        result["sha256"] = sha256_file(path)
        if require_resumable and not result["resumable"]:
            result["error"] = "checkpoint is loadable but has no optimizer/epoch resume state"
    except Exception as exc:
        result["error"] = repr(exc)
    return result


def read_results_csv(run_dir: Path) -> dict[str, Any]:
    path = run_dir / "results.csv"
    if not path.exists() or path.stat().st_size == 0:
        return {"exists": False, "rows": 0, "last_epoch": None, "path": str(path)}
    try:
        frame = pd.read_csv(path)
        epoch_column = next((column for column in frame.columns if column.strip() == "epoch"), None)
        last_epoch = int(frame[epoch_column].dropna().iloc[-1]) if epoch_column and not frame.empty else len(frame) - 1
        return {
            "exists": True,
            "rows": len(frame),
            "last_epoch": last_epoch,
            "columns": list(frame.columns),
            "path": str(path),
        }
    except Exception as exc:
        return {"exists": True, "rows": 0, "last_epoch": None, "path": str(path), "error": repr(exc)}


def find_best_last(run_dir: Path) -> tuple[Path | None, Path | None]:
    best = run_dir / "weights" / "best.pt"
    last = run_dir / "weights" / "last.pt"
    return best if best.exists() else None, last if last.exists() else None


def marker_valid(
    marker_path: Path,
    expected_signature: str,
    artifact_path: Path,
    run_dir: Path,
) -> tuple[bool, dict[str, Any]]:
    marker = read_json(marker_path, {}) or {}
    reasons: list[str] = []
    if marker.get("status") != "completed":
        reasons.append("marker status is not completed")
    if marker.get("job_signature") != expected_signature:
        reasons.append("job signature mismatch")
    if marker.get("model_key") != MODEL_KEY:
        reasons.append("model key mismatch")
    if str(marker.get("loss_mode", "")).lower() != EXPECTED_LOSS:
        reasons.append("loss mode mismatch")
    if Path(str(marker.get("run_dir", "."))).resolve() != run_dir.resolve():
        reasons.append("run directory mismatch")
    if Path(str(marker.get("artifact", "."))).resolve() != artifact_path.resolve():
        reasons.append("artifact path mismatch")

    checkpoint = validate_checkpoint(artifact_path)
    if not checkpoint.get("loadable"):
        reasons.append(f"artifact checkpoint invalid: {checkpoint.get('error')}")
    if not marker.get("artifact_sha256") or marker.get("artifact_sha256") != checkpoint.get("sha256"):
        reasons.append("artifact SHA-256 missing or mismatched")

    results = read_results_csv(run_dir)
    if not results.get("exists") or results.get("rows", 0) < 1:
        reasons.append("results.csv missing or empty")

    loss_audit = read_json(run_dir / "loss_audit_fixed.json", {}) or {}
    if loss_audit.get("matches_expected") is not True:
        reasons.append("loss audit does not confirm matches_expected=true")
    if str(loss_audit.get("loss_mode", "")).lower() != EXPECTED_LOSS:
        reasons.append("loss audit is missing or not SIoU")

    return not reasons, {
        "marker": marker,
        "checkpoint": checkpoint,
        "results": results,
        "loss_audit": loss_audit,
        "reasons": reasons,
    }


def add_callbacks(model: Any, job_id: str, run_dir: Path, heartbeat: Path, events: Path) -> None:
    state = {"loss_written": False}

    def scalar(value: Any) -> Any:
        try:
            if hasattr(value, "item"):
                return value.item()
            if isinstance(value, (str, int, float, bool)) or value is None:
                return value
            return str(value)
        except Exception:
            return str(value)

    def emit(event: str, trainer: Any) -> None:
        payload: dict[str, Any] = {
            "time": now_iso(),
            "event": event,
            "job_id": job_id,
            "model_key": MODEL_KEY,
            "run_dir": str(run_dir),
        }
        for name in ("epoch", "best_fitness", "fitness"):
            value = getattr(trainer, name, None)
            if value is not None:
                payload[name] = scalar(value)
        metrics = getattr(trainer, "metrics", None)
        if isinstance(metrics, dict):
            payload["metrics"] = {str(key): scalar(value) for key, value in metrics.items()}
        append_jsonl(events, payload)
        atomic_write_json(heartbeat, payload)

    def audit_loss(trainer: Any) -> None:
        if state["loss_written"]:
            return
        train_model = getattr(trainer, "model", None)
        criterion = getattr(train_model, "criterion", None)
        if criterion is None:
            return
        bbox_loss = getattr(criterion, "bbox_loss", None)
        criterion_name = criterion.__class__.__name__
        bbox_name = bbox_loss.__class__.__name__ if bbox_loss is not None else ""
        inferred = "siou" if "siou" in (criterion_name + bbox_name).lower() else "ciou"
        payload = {
            "time": now_iso(),
            "job_id": job_id,
            "model_key": MODEL_KEY,
            "expected_loss": EXPECTED_LOSS,
            "loss_mode": inferred,
            "criterion_class": criterion_name,
            "bbox_loss_class": bbox_name,
            "matches_expected": inferred == EXPECTED_LOSS,
        }
        atomic_write_json(run_dir / "loss_audit_fixed.json", payload)
        append_jsonl(events, {"event": "loss_audit", **payload})
        state["loss_written"] = True
        if inferred != EXPECTED_LOSS:
            raise RuntimeError(f"Loss audit failed: {payload}")

    callbacks = {
        "on_train_start": lambda trainer: emit("on_train_start", trainer),
        "on_train_epoch_end": lambda trainer: emit("on_train_epoch_end", trainer),
        "on_fit_epoch_end": lambda trainer: emit("on_fit_epoch_end", trainer),
        "on_model_save": lambda trainer: emit("on_model_save", trainer),
        "on_train_end": lambda trainer: emit("on_train_end", trainer),
        "on_train_batch_end": audit_loss,
    }
    for event, callback in callbacks.items():
        model.add_callback(event, callback)


# =============================================================================
# 5. MODEL RUNTIME AUDIT
# =============================================================================


def check_ultralytics_version() -> str:
    import ultralytics

    version = str(ultralytics.__version__)
    if env_bool("STRICT_ULTRALYTICS_VERSION", True) and version != EXPECTED_ULTRALYTICS_VERSION:
        raise RuntimeError(
            f"This script was validated for ultralytics=={EXPECTED_ULTRALYTICS_VERSION}, found {version}. "
            "Install the pinned version or explicitly set STRICT_ULTRALYTICS_VERSION=0 after reviewing APIs."
        )
    return version


def run_model_audit(ultralytics_version: str) -> dict[str, Any]:
    import torch
    from ultralytics import YOLO
    from ultralytics.cfg import get_cfg
    from ultralytics.utils.loss import BboxLoss
    from ultralytics.utils.torch_utils import get_flops, get_num_params

    register_fixed_modules()
    from models.fruitnet_gcs_fixed.siou_loss import (
        SIoUBboxLossFixed,
        SIoUDetectionLossFixed,
        siou_loss_xyxy_fixed,
    )
    from models.fruitnet_gcs_fixed.siou_trainer import (
        SIoUDetectionModelFixed,
        SIoUDetectionTrainerFixed,
    )

    expected_bbox_parameters = [
        "self", "pred_dist", "pred_bboxes", "anchor_points", "target_bboxes",
        "target_scores", "target_scores_sum", "fg_mask", "imgsz", "stride",
    ]
    actual_bbox_parameters = list(inspect.signature(BboxLoss.forward).parameters)
    if actual_bbox_parameters != expected_bbox_parameters:
        raise RuntimeError(
            "Ultralytics BboxLoss.forward API differs from the validated API. "
            f"Expected {expected_bbox_parameters}, found {actual_bbox_parameters}."
        )

    pred = torch.tensor(
        [[0.1, 0.1, 0.5, 0.5], [0.2, 0.2, 0.6, 0.7]],
        dtype=torch.float32,
        requires_grad=True,
    )
    target = torch.tensor(
        [[0.1, 0.1, 0.5, 0.5], [0.25, 0.22, 0.62, 0.72]],
        dtype=torch.float32,
    )
    loss = siou_loss_xyxy_fixed(pred, target)
    loss.sum().backward()
    if float(loss[0].detach()) > 1e-5 or not torch.isfinite(loss).all():
        raise RuntimeError("SIoU numerical smoke test failed")
    if pred.grad is None or not torch.isfinite(pred.grad).all():
        raise RuntimeError("SIoU gradient smoke test failed")

    architecture_model = YOLO(str(MODEL_YAML))
    core = architecture_model.model
    parameters = int(get_num_params(core))
    imgsz = env_int("MODEL_INFO_IMG_SIZE", env_int("IMG_SIZE", 640))
    gflops = get_flops(core, imgsz=imgsz)
    if not gflops:
        raise RuntimeError("GFLOPs calculation failed; install ultralytics-thop")

    custom_model = SIoUDetectionModelFixed(str(MODEL_YAML), nc=80, ch=3, verbose=False)
    custom_model.args = get_cfg()
    criterion = custom_model.init_criterion()
    connected = isinstance(criterion, SIoUDetectionLossFixed) and isinstance(criterion.bbox_loss, SIoUBboxLossFixed)
    if not connected:
        raise RuntimeError("FruitNet-GCS fixed did not instantiate SIoUDetectionLossFixed/SIoUBboxLossFixed")

    audit = {
        "schema_version": PIPELINE_SCHEMA_VERSION,
        "created_at": now_iso(),
        "model_key": MODEL_KEY,
        "model_yaml": str(MODEL_YAML),
        "model_yaml_sha256": sha256_file(MODEL_YAML),
        "ultralytics_version": ultralytics_version,
        "validated_ultralytics_version": EXPECTED_ULTRALYTICS_VERSION,
        "parameters_nc80": parameters,
        "gflops_nc80": float(gflops),
        "gflops_imgsz": imgsz,
        "loss_mode": EXPECTED_LOSS,
        "criterion_class": criterion.__class__.__name__,
        "bbox_loss_class": criterion.bbox_loss.__class__.__name__,
        "trainer_class": f"{SIoUDetectionTrainerFixed.__module__}.{SIoUDetectionTrainerFixed.__name__}",
        "siou_connected": connected,
        "bbox_forward_parameters": actual_bbox_parameters,
        "note": "Same parameters/GFLOPs as GC is expected because SIoU changes training loss, not architecture.",
    }
    atomic_write_json(MODEL_AUDIT_DIR / "fruitnet_gcs_fixed_model_audit.json", audit)
    return audit


# =============================================================================
# 6. GENERIC TRAIN AND EVALUATION JOBS
# =============================================================================


def script_sha256() -> str:
    if "__file__" in globals():
        path = Path(__file__).resolve()
        if path.exists():
            return sha256_file(path)
    return stable_hash({"model_yaml": MODEL_YAML_SOURCE, "siou": SIOU_LOSS_SOURCE, "trainer": SIOU_TRAINER_SOURCE})


def helper_hashes() -> dict[str, str]:
    return {
        "coord_attention_sha256": sha256_file(MODEL_PACKAGE_DIR / "coord_attention.py"),
        "siou_loss_sha256": sha256_file(MODEL_PACKAGE_DIR / "siou_loss.py"),
        "siou_trainer_sha256": sha256_file(MODEL_PACKAGE_DIR / "siou_trainer.py"),
        "register_sha256": sha256_file(MODEL_PACKAGE_DIR / "register.py"),
        "model_yaml_sha256": sha256_file(MODEL_YAML),
        "pipeline_script_sha256": script_sha256(),
    }


def train_job(
    *,
    job_id: str,
    stage: str,
    data_yaml: Path,
    dataset_manifest_hash: str,
    source: Path,
    source_is_checkpoint: bool,
    train_cfg: Mapping[str, Any],
    runs_root: Path,
    artifact: Path,
    marker: Path,
    force: bool,
) -> dict[str, Any]:
    from ultralytics import YOLO
    from models.fruitnet_gcs_fixed.siou_trainer import SIoUDetectionTrainerFixed

    source_payload = (
        {"path": str(source.resolve()), "sha256": sha256_file(source)}
        if source.exists()
        else {"path": str(source), "missing": True}
    )
    if source_is_checkpoint:
        source_check = validate_checkpoint(source)
        if not source_check.get("loadable"):
            raise RuntimeError(f"Source checkpoint is invalid: {source_check}")

    signature_payload = {
        "schema_version": PIPELINE_SCHEMA_VERSION,
        "stage": stage,
        "job_id": job_id,
        "model_key": MODEL_KEY,
        "loss_mode": EXPECTED_LOSS,
        "dataset_yaml": str(data_yaml.resolve()),
        "dataset_manifest_hash": dataset_manifest_hash,
        "source": source_payload,
        "source_is_checkpoint": source_is_checkpoint,
        "train_cfg": dict(train_cfg),
        "ultralytics": EXPECTED_ULTRALYTICS_VERSION,
        "code": helper_hashes(),
    }
    job_signature = stable_hash(signature_payload)
    run_dir = runs_root / f"{job_id}__{job_signature[:12]}"
    heartbeat = marker.with_name(marker.stem.replace(".done", "") + ".heartbeat_fixed.json")
    events = marker.with_name(marker.stem.replace(".done", "") + ".events_fixed.jsonl")
    failure = marker.with_name(marker.stem.replace(".done", "") + ".failed_fixed.json")

    valid, evidence = marker_valid(marker, job_signature, artifact, run_dir)
    if valid and not force:
        logging.info("SKIP valid completed training job: %s", job_id)
        return {
            "job_id": job_id,
            "stage": stage,
            "status": "skipped_valid",
            "run_dir": str(run_dir),
            "artifact": str(artifact),
            "artifact_sha256": evidence.get("checkpoint", {}).get("sha256"),
            "job_signature": job_signature,
            "loss_mode": EXPECTED_LOSS,
            "evidence": evidence,
        }

    best, last = find_best_last(run_dir)

    # Recover a run that finished training but crashed while exporting the artifact/marker.
    # Ultralytics strips optimizer state and sets epoch=-1 only after a successful finalization.
    if not force and best is not None and last is not None:
        best_check = validate_checkpoint(best)
        last_check = validate_checkpoint(last)
        results = read_results_csv(run_dir)
        loss_audit = read_json(run_dir / "loss_audit_fixed.json", {}) or {}
        orphan_finished = bool(
            best_check.get("loadable")
            and last_check.get("loadable")
            and not last_check.get("resumable")
            and int(last_check.get("epoch", -999) or -999) == -1
            and results.get("rows", 0) >= 1
            and loss_audit.get("matches_expected") is True
            and loss_audit.get("loss_mode") == EXPECTED_LOSS
        )
        if orphan_finished:
            atomic_copy(best, artifact)
            artifact_check = validate_checkpoint(artifact)
            if not artifact_check.get("loadable"):
                raise RuntimeError(f"Recovered artifact is invalid: {artifact_check}")
            done = {
                "schema_version": PIPELINE_SCHEMA_VERSION,
                "status": "completed",
                "completed_at": now_iso(),
                "job_id": job_id,
                "model_key": MODEL_KEY,
                "stage": stage,
                "job_signature": job_signature,
                "signature_payload": signature_payload,
                "loss_mode": EXPECTED_LOSS,
                "run_dir": str(run_dir),
                "best_pt": str(best),
                "last_pt": str(last),
                "artifact": str(artifact),
                "artifact_sha256": artifact_check.get("sha256"),
                "results": results,
                "loss_audit": loss_audit,
                "resumed": False,
                "recovered_after_marker_crash": True,
                "log_file": str(LOG_PATH),
            }
            atomic_write_json(marker, done)
            append_jsonl(events, {"time": now_iso(), "event": "job_recovered_completed", **done})
            logging.info("RECOVERED completed run without retraining: %s", job_id)
            return {
                "job_id": job_id,
                "stage": stage,
                "status": "recovered_completed",
                "run_dir": str(run_dir),
                "artifact": str(artifact),
                "artifact_sha256": artifact_check.get("sha256"),
                "job_signature": job_signature,
                "loss_mode": EXPECTED_LOSS,
                "epochs_recorded": results.get("rows"),
                "resumed": False,
            }

    resume = bool(last and validate_checkpoint(last, require_resumable=True).get("resumable"))
    if not resume:
        stale = archive_stale_run(
            run_dir,
            "Marker invalid and no resumable last.pt exists; starting a clean fixed run.",
        )
        if stale:
            logging.warning("Archived stale fixed run: %s -> %s", run_dir, stale)
    ensure_dirs([run_dir, marker.parent, artifact.parent])

    logging.info(
        "START %s | stage=%s | resume=%s | source=%s | run_dir=%s",
        job_id,
        stage,
        resume,
        source,
        run_dir,
    )
    append_jsonl(
        events,
        {
            "time": now_iso(),
            "event": "job_start",
            "job_id": job_id,
            "stage": stage,
            "resume": resume,
            "job_signature": job_signature,
            "run_dir": str(run_dir),
        },
    )

    try:
        if failure.exists():
            failure.unlink()
        model = YOLO(str(last if resume else source))
        add_callbacks(model, job_id, run_dir, heartbeat, events)

        if resume:
            model.train(trainer=SIoUDetectionTrainerFixed, resume=True)
        else:
            model.train(
                trainer=SIoUDetectionTrainerFixed,
                data=str(data_yaml.resolve()),
                epochs=int(train_cfg["epochs"]),
                imgsz=int(train_cfg["imgsz"]),
                batch=int(train_cfg["batch"]),
                workers=int(train_cfg["workers"]),
                seed=int(train_cfg["seed"]),
                patience=int(train_cfg["patience"]),
                save_period=int(train_cfg["save_period"]),
                device=str(train_cfg["device"]),
                project=str(runs_root),
                name=run_dir.name,
                exist_ok=True,
                pretrained=bool(source_is_checkpoint),
                save=True,
                plots=bool(train_cfg.get("plots", True)),
                verbose=True,
            )

        best, last = find_best_last(run_dir)
        if best is None or last is None:
            raise RuntimeError("Training returned without both best.pt and last.pt")
        best_check = validate_checkpoint(best)
        last_check = validate_checkpoint(last)
        results = read_results_csv(run_dir)
        loss_audit = read_json(run_dir / "loss_audit_fixed.json", {}) or {}
        if not best_check.get("loadable") or not last_check.get("loadable"):
            raise RuntimeError(f"Checkpoint validation failed: best={best_check}; last={last_check}")
        if results.get("rows", 0) < 1:
            raise RuntimeError("results.csv is missing or empty")
        if loss_audit.get("matches_expected") is not True or loss_audit.get("loss_mode") != EXPECTED_LOSS:
            raise RuntimeError(f"Runtime SIoU audit failed: {loss_audit}")

        atomic_copy(best, artifact)
        artifact_check = validate_checkpoint(artifact)
        if not artifact_check.get("loadable"):
            raise RuntimeError(f"Exported artifact is invalid: {artifact_check}")

        done = {
            "schema_version": PIPELINE_SCHEMA_VERSION,
            "status": "completed",
            "completed_at": now_iso(),
            "job_id": job_id,
            "model_key": MODEL_KEY,
            "stage": stage,
            "job_signature": job_signature,
            "signature_payload": signature_payload,
            "loss_mode": EXPECTED_LOSS,
            "run_dir": str(run_dir),
            "best_pt": str(best),
            "last_pt": str(last),
            "artifact": str(artifact),
            "artifact_sha256": artifact_check.get("sha256"),
            "results": results,
            "loss_audit": loss_audit,
            "resumed": resume,
            "log_file": str(LOG_PATH),
        }
        atomic_write_json(marker, done)
        append_jsonl(events, {"time": now_iso(), "event": "job_completed", **done})
        return {
            "job_id": job_id,
            "stage": stage,
            "status": "completed",
            "run_dir": str(run_dir),
            "artifact": str(artifact),
            "artifact_sha256": artifact_check.get("sha256"),
            "job_signature": job_signature,
            "loss_mode": EXPECTED_LOSS,
            "epochs_recorded": results.get("rows"),
            "resumed": resume,
        }
    except Exception as exc:
        write_failure(
            failure,
            job_id,
            exc,
            {"stage": stage, "run_dir": str(run_dir), "job_signature": job_signature},
        )
        logging.exception("Training failed for %s", job_id)
        raise


def metrics_to_dict(metrics: Any) -> dict[str, Any]:
    box = getattr(metrics, "box", None)
    speed = getattr(metrics, "speed", {}) or {}
    result = {
        "precision": float(getattr(box, "mp", float("nan"))) if box is not None else float("nan"),
        "recall": float(getattr(box, "mr", float("nan"))) if box is not None else float("nan"),
        "map50": float(getattr(box, "map50", float("nan"))) if box is not None else float("nan"),
        "map75": float(getattr(box, "map75", float("nan"))) if box is not None else float("nan"),
        "map50_95": float(getattr(box, "map", float("nan"))) if box is not None else float("nan"),
        "fitness": float(getattr(metrics, "fitness", float("nan"))),
    }
    for key, value in speed.items():
        try:
            result[f"speed_{key}_ms"] = float(value)
        except Exception:
            pass
    return result


def eval_marker_valid(
    marker: Path,
    expected_signature: str,
    metrics_path: Path,
    eval_split: str,
) -> tuple[bool, dict[str, Any]]:
    payload = read_json(marker, {}) or {}
    metrics = read_json(metrics_path, {}) or {}
    reasons: list[str] = []
    if payload.get("status") != "completed":
        reasons.append("marker status is not completed")
    if payload.get("eval_signature") != expected_signature:
        reasons.append("eval signature mismatch")
    if payload.get("evaluation_split") != eval_split:
        reasons.append("evaluation split mismatch")
    if Path(str(payload.get("metrics_path", "."))).resolve() != metrics_path.resolve():
        reasons.append("metrics path mismatch")
    if metrics.get("eval_signature") != expected_signature:
        reasons.append("metrics signature mismatch")
    if metrics.get("evaluation_split") != eval_split:
        reasons.append("metrics evaluation split mismatch")
    if not metrics_path.exists() or payload.get("metrics_sha256") != sha256_file(metrics_path):
        reasons.append("metrics SHA-256 missing or mismatched")
    return not reasons, {"marker": payload, "metrics": metrics, "reasons": reasons}


def evaluate_job(
    *,
    dataset: str,
    data_yaml: Path,
    dataset_manifest_hash: str,
    eval_split: str,
    trained_artifact: Path,
    train_signature: str,
    cfg: Mapping[str, Any],
    force: bool,
) -> dict[str, Any]:
    from ultralytics import YOLO
    from ultralytics.utils.torch_utils import get_flops, get_num_params

    checkpoint = validate_checkpoint(trained_artifact)
    if not checkpoint.get("loadable"):
        raise RuntimeError(f"Cannot evaluate invalid checkpoint: {checkpoint}")

    job_id = f"{dataset}__fruitnet_gcs_fixed"
    eval_signature_payload = {
        "schema_version": PIPELINE_SCHEMA_VERSION,
        "stage": "public_benchmark_eval_fixed",
        "job_id": job_id,
        "model_key": MODEL_KEY,
        "dataset": dataset,
        "dataset_manifest_hash": dataset_manifest_hash,
        "trained_checkpoint_sha256": checkpoint.get("sha256"),
        "train_signature": train_signature,
        "evaluation_split": eval_split,
        "imgsz": int(cfg["imgsz"]),
        "batch": int(cfg["batch"]),
        "workers": int(cfg["workers"]),
        "device": str(cfg["device"]),
        "code": helper_hashes(),
    }
    eval_signature = stable_hash(eval_signature_payload)
    marker = BENCH_MARKER_ROOT / f"{job_id}.eval.done_fixed.json"
    metrics_path = BENCH_METRICS_ROOT / f"{job_id}.{eval_split}_metrics_fixed.json"
    failure = BENCH_MARKER_ROOT / f"{job_id}.eval.failed_fixed.json"

    valid, evidence = eval_marker_valid(marker, eval_signature, metrics_path, eval_split)
    if valid and not force:
        logging.info("SKIP valid completed evaluation: %s", job_id)
        metrics = read_json(metrics_path, {}) or {}
        return {"status": "skipped_valid", **metrics, "evidence": evidence}

    try:
        if failure.exists():
            failure.unlink()
        model = YOLO(str(trained_artifact))
        metrics = model.val(
            data=str(data_yaml.resolve()),
            split=eval_split,
            imgsz=int(cfg["imgsz"]),
            batch=int(cfg["batch"]),
            workers=int(cfg["workers"]),
            device=str(cfg["device"]),
            project=str(BENCH_EVAL_RUNS_ROOT / dataset),
            name=f"fruitnet_gcs_fixed__{eval_signature[:12]}",
            exist_ok=True,
            plots=bool(cfg.get("plots", True)),
            verbose=True,
        )
        result = metrics_to_dict(metrics)
        result.update(
            {
                "schema_version": PIPELINE_SCHEMA_VERSION,
                "created_at": now_iso(),
                "status": "completed",
                "dataset": dataset,
                "model_key": MODEL_KEY,
                "evaluation_split": eval_split,
                "independent_test": eval_split == "test",
                "loss_mode_used_during_training": EXPECTED_LOSS,
                "trained_checkpoint": str(trained_artifact),
                "trained_checkpoint_sha256": checkpoint.get("sha256"),
                "dataset_manifest_hash": dataset_manifest_hash,
                "train_signature": train_signature,
                "eval_signature": eval_signature,
                "parameters": int(get_num_params(model.model)),
                "gflops": float(get_flops(model.model, imgsz=int(cfg["imgsz"])) or 0.0),
            }
        )
        atomic_write_json(metrics_path, result)
        atomic_write_json(
            marker,
            {
                "schema_version": PIPELINE_SCHEMA_VERSION,
                "status": "completed",
                "completed_at": now_iso(),
                "job_id": job_id,
                "eval_signature": eval_signature,
                "evaluation_split": eval_split,
                "metrics_path": str(metrics_path),
                "metrics_sha256": sha256_file(metrics_path),
            },
        )
        return result
    except Exception as exc:
        write_failure(failure, job_id, exc, {"eval_signature": eval_signature, "evaluation_split": eval_split})
        logging.exception("Evaluation failed for %s", job_id)
        raise


# =============================================================================
# 7. MAIN PIPELINE
# =============================================================================


def main() -> None:
    os.chdir(PROJECT_ROOT)
    ensure_dirs(
        [
            FIXED_OUTPUT_ROOT,
            MODEL_PACKAGE_DIR,
            MODEL_YAML.parent,
            FIXED_DATA_CONFIG_DIR,
            MODEL_AUDIT_DIR,
            COCO_RUNS_ROOT,
            COCO_ARTIFACT_ROOT,
            COCO_RESULT_ROOT,
            COCO_MARKER_ROOT,
            BENCH_RUNS_ROOT,
            BENCH_EVAL_RUNS_ROOT,
            BENCH_ARTIFACT_ROOT,
            BENCH_RESULT_ROOT,
            BENCH_MARKER_ROOT,
            BENCH_METRICS_ROOT,
        ]
    )
    setup_logging()
    logging.info("PROJECT_ROOT=%s", PROJECT_ROOT)
    logging.info("DATA_ROOT=%s", DATA_ROOT)
    logging.info("OUTPUT_ROOT=%s", OUTPUT_ROOT)
    logging.info("FIXED_OUTPUT_ROOT=%s", FIXED_OUTPUT_ROOT)
    logging.info("Python=%s", sys.version)
    logging.info("Platform=%s", platform.platform())
    logging.info("Log=%s", LOG_PATH)

    install_fixed_model_files()
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    try:
        import torch
        import ultralytics
    except ImportError as exc:
        raise RuntimeError(
            "Install the validated runtime first: "
            "pip install ultralytics==8.4.60 ultralytics-thop pandas PyYAML"
        ) from exc

    ultralytics_version = check_ultralytics_version()
    register_fixed_modules()
    model_audit = run_model_audit(ultralytics_version)
    logging.info("Model audit OK: %s", json.dumps(model_audit, ensure_ascii=False))

    coco_yaml, _ = build_fixed_dataset_yaml("coco")
    coco_audit = audit_yolo_dataset(coco_yaml, ["train", "val"])
    atomic_write_json(COCO_RESULT_ROOT / "coco_dataset_audit_fixed.json", coco_audit)
    if not coco_audit["valid"]:
        raise RuntimeError(f"COCO audit failed: {coco_audit['errors']}")

    common_cfg = {
        "imgsz": env_int("IMG_SIZE", 640),
        "workers": env_int("WORKERS", 8),
        "seed": env_int("SEED", 42),
        "save_period": env_int("SAVE_PERIOD", 1),
        "device": os.environ.get("DEVICE", "0"),
        "plots": env_bool("PLOTS", True),
    }
    coco_cfg = {
        **common_cfg,
        "epochs": env_int("COCO_EPOCHS", 50),
        "batch": env_int("COCO_BATCH", 4),
        "patience": env_int("COCO_PATIENCE", 20),
    }
    benchmark_cfg = {
        **common_cfg,
        "epochs": env_int("BENCHMARK_EPOCHS", 100),
        "batch": env_int("BENCHMARK_BATCH", 8),
        "patience": env_int("BENCHMARK_PATIENCE", 30),
    }

    rows: list[dict[str, Any]] = []

    if env_bool("RUN_PRETRAIN", True):
        pretrain_row = train_job(
            job_id="fruitnet_gcs_fixed_coco",
            stage="coco_pretrain_fixed",
            data_yaml=coco_yaml,
            dataset_manifest_hash=coco_audit["manifest_hash"],
            source=MODEL_YAML,
            source_is_checkpoint=False,
            train_cfg=coco_cfg,
            runs_root=COCO_RUNS_ROOT,
            artifact=COCO_ARTIFACT,
            marker=COCO_MARKER_ROOT / "fruitnet_gcs_fixed_coco.done_fixed.json",
            force=env_bool("FORCE_PRETRAIN", False),
        )
        rows.append(pretrain_row)
        atomic_write_csv(COCO_RESULT_ROOT / "coco_pretrain_progress_fixed.csv", pd.DataFrame(rows))
    else:
        checkpoint = validate_checkpoint(COCO_ARTIFACT)
        if not checkpoint.get("loadable"):
            raise RuntimeError(
                "RUN_PRETRAIN=0 but fruitnet_gcs_fixed COCO artifact is missing or invalid: "
                f"{checkpoint}"
            )
        rows.append(
            {
                "job_id": "fruitnet_gcs_fixed_coco",
                "stage": "coco_pretrain_fixed",
                "status": "skipped_by_configuration",
                "artifact": str(COCO_ARTIFACT),
                "artifact_sha256": checkpoint.get("sha256"),
            }
        )

    if not env_bool("RUN_BENCHMARK", True):
        logging.info("RUN_BENCHMARK=0; stopping after COCO pretraining.")
        atomic_write_csv(COCO_RESULT_ROOT / "coco_pretrain_summary_fixed.csv", pd.DataFrame(rows))
        return

    selected_datasets = [
        item.strip().lower()
        for item in os.environ.get("BENCHMARK_DATASETS", "minneapple,gwhd").split(",")
        if item.strip()
    ]
    unknown = [item for item in selected_datasets if item not in {"minneapple", "gwhd"}]
    if unknown:
        raise ValueError(f"Unsupported BENCHMARK_DATASETS: {unknown}")

    benchmark_rows: list[dict[str, Any]] = []
    for dataset in selected_datasets:
        data_yaml, eval_split = build_fixed_dataset_yaml(dataset)
        required_splits = ["train", "val"] + (["test"] if eval_split == "test" else [])
        audit = audit_yolo_dataset(data_yaml, required_splits)
        atomic_write_json(BENCH_RESULT_ROOT / f"{dataset}_dataset_audit_fixed.json", audit)
        if not audit["valid"]:
            raise RuntimeError(f"{dataset} audit failed: {audit['errors']}")

        job_id = f"{dataset}__fruitnet_gcs_fixed"
        trained_artifact = BENCH_ARTIFACT_ROOT / dataset / f"fruitnet_gcs_fixed_{dataset}_best.pt"
        train_marker = BENCH_MARKER_ROOT / f"{job_id}.train.done_fixed.json"
        train_row = train_job(
            job_id=job_id,
            stage="public_benchmark_train_fixed",
            data_yaml=data_yaml,
            dataset_manifest_hash=audit["manifest_hash"],
            source=COCO_ARTIFACT,
            source_is_checkpoint=True,
            train_cfg=benchmark_cfg,
            runs_root=BENCH_RUNS_ROOT / dataset,
            artifact=trained_artifact,
            marker=train_marker,
            force=env_bool("FORCE_BENCHMARK", False),
        )

        eval_row: dict[str, Any] = {
            "status": "not_run",
            "evaluation_split": eval_split,
        }
        if env_bool("RUN_EVAL", True):
            eval_row = evaluate_job(
                dataset=dataset,
                data_yaml=data_yaml,
                dataset_manifest_hash=audit["manifest_hash"],
                eval_split=eval_split,
                trained_artifact=trained_artifact,
                train_signature=str(train_row["job_signature"]),
                cfg=benchmark_cfg,
                force=env_bool("FORCE_EVAL", False),
            )

        combined = {
            "dataset": dataset,
            "model_key": MODEL_KEY,
            "loss_mode": EXPECTED_LOSS,
            "train_status": train_row.get("status"),
            "eval_status": eval_row.get("status"),
            "evaluation_split": eval_row.get("evaluation_split", eval_split),
            "independent_test": eval_row.get("independent_test", eval_split == "test"),
            "precision": eval_row.get("precision"),
            "recall": eval_row.get("recall"),
            "map50": eval_row.get("map50"),
            "map75": eval_row.get("map75"),
            "map50_95": eval_row.get("map50_95"),
            "parameters": eval_row.get("parameters"),
            "gflops": eval_row.get("gflops"),
            "run_dir": train_row.get("run_dir"),
            "artifact": train_row.get("artifact"),
            "artifact_sha256": train_row.get("artifact_sha256"),
            "train_signature": train_row.get("job_signature"),
            "dataset_manifest_hash": audit["manifest_hash"],
        }
        benchmark_rows.append(combined)
        atomic_write_csv(BENCH_RESULT_ROOT / "benchmark_progress_fixed.csv", pd.DataFrame(benchmark_rows))

    atomic_write_csv(COCO_RESULT_ROOT / "coco_pretrain_summary_fixed.csv", pd.DataFrame(rows))
    atomic_write_csv(BENCH_RESULT_ROOT / "benchmark_summary_fixed.csv", pd.DataFrame(benchmark_rows))
    atomic_write_json(
        FIXED_OUTPUT_ROOT / "fruitnet_gcs_fixed_pipeline_status.json",
        {
            "schema_version": PIPELINE_SCHEMA_VERSION,
            "created_at": now_iso(),
            "status": "completed",
            "model_key": MODEL_KEY,
            "loss_mode": EXPECTED_LOSS,
            "ultralytics_version": ultralytics_version,
            "project_root": str(PROJECT_ROOT),
            "data_root": str(DATA_ROOT),
            "fixed_output_root": str(FIXED_OUTPUT_ROOT),
            "model_audit": model_audit,
            "coco_rows": rows,
            "benchmark_rows": benchmark_rows,
            "log_file": str(LOG_PATH),
        },
    )

    print("\n=== COCO PRETRAIN FIXED ===")
    print(pd.DataFrame(rows).to_string(index=False))
    print("\n=== PUBLIC BENCHMARK FIXED ===")
    print(pd.DataFrame(benchmark_rows).to_string(index=False))
    print(f"\nCompleted. All new artifacts are isolated under: {FIXED_OUTPUT_ROOT}")


if __name__ == "__main__":
    main()

2026-07-11 19:05:12,895 | INFO | PROJECT_ROOT=/home/diy-hus/fruitnet-chili-yield
2026-07-11 19:05:12,897 | INFO | DATA_ROOT=/mnt/f/fruitnet-chili-yield-data
2026-07-11 19:05:12,897 | INFO | OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
2026-07-11 19:05:12,898 | INFO | FIXED_OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs/fruitnet_gcs_fixed
2026-07-11 19:05:12,898 | INFO | Python=3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
2026-07-11 19:05:12,900 | INFO | Platform=Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
2026-07-11 19:05:12,901 | INFO | Log=/mnt/f/fruitnet-chili-yield-outputs/fruitnet_gcs_fixed/logs_fixed/fruitnet_gcs_fixed_20260711_190512.log
2026-07-11 19:05:18,361 | INFO | Model audit OK: {"schema_version": 3, "created_at": "2026-07-11T19:05:18.357383+00:00", "model_key": "fruitnet_gcs_fixed", "model_yaml": "/home/diy-hus/fruitnet-chili-yield/configs/models/fruitnet_gcs_fixed.yaml", "model_yaml_sha256": "728ce53ffb60a0819a45dd1e7de683481f522f65c8c839a19d9

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      1.69G      2.359      4.097       2.64          3        640: 100% ━━━━━━━━━━━━ 29572/29572 4.2it/s 1:57:18<0.2ss5
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 625/625 5.7it/s 1:49<0.1sss
                   all       5000      36334      0.274      0.133      0.086     0.0497

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      1.76G      1.515      2.404      1.692          4        640: 0% ──────────── 0/29572  0.3s

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      1.77G      1.634      2.831      1.797          3        640: 100% ━━━━━━━━━━━━ 29572/29572 5.5it/s 1:29:01<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 625/625 10.5it/s 59.8s.3sss
                   all       5000      36334      0.334      0.218       0.18      0.113

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      1.77G      1.282      2.796      1.437          4        640: 0% ──────────── 0/29572  0.3s

...

     98/100      3.93G      1.461     0.6209     0.8548          8        640: 100% ━━━━━━━━━━━━ 67/67 4.2it/s 16.1s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 6.5it/s 1.4s0.2s
                   all        134       5585      0.863      0.803      0.859      0.425

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     99/100      3.93G      1.467     0.6295     0.8182          8        640: 0% ──────────── 0/67  0.2s

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      3.93G      1.442     0.6053     0.8525          8        640: 100% ━━━━━━━━━━━━ 67/67 4.0it/s 16.8s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 6.5it/s 1.4s0.2s
                   all        134       5585       0.87        0.8      0.862      0.425

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    100/100      3.93G      1.463     0.6073     0.8615          8        640: 0% ──────────── 0/67  0.3s

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    100/100      3.93G      1.454     0.6128     0.8487          8        640: 100% ━━━━━━━━━━━━ 67/67 4.1it/s 16.3s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 6.5it/s 1.4s0.2s
                   all        134       5585      0.864      0.804      0.858      0.425

100 epochs completed in 0.561 hours.
Optimizer stripped from /mnt/f/fruitnet-chili-yield-outputs/fruitnet_gcs_fixed/runs/public_benchmark_fixed/minneapple/minneapple__fruitnet_gcs_fixed__a57d237298b9/weights/last.pt, 28.1MB
Optimizer stripped from /mnt/f/fruitnet-chili-yield-outputs/fruitnet_gcs_fixed/runs/public_benchmark_fixed/minneapple/minneapple__fruitnet_gcs_fixed__a57d237298b9/weights/best.pt, 28.1MB

Validating /mnt/f/fruitnet-chili-yield-outputs/fruitnet_gcs_fixed/runs/public_benchmark_fixed/minneapple/minneapple__fruitnet_gcs_fixed__a57d237298b9/weights/best.pt...
Ultralytics 8.4.60 🚀 Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce R